# Potato Yield vs. Weather Modelling (weekly features, 5–10 selected)

**Growing season:** Standard Meteorological Weeks (SMW) 42–52 (mid-Oct–Dec) and SMW 1–8 (Jan–Feb), the Rabi potato season.

This notebook follows the workflow below (matches the process flow diagram):

1. Install required python libraries
2. Upload the weather & potato yield data files
3. Import libraries & set up analysis
4. Load daily weather data & convert it to SMW
5. Load potato yield data & define the potato growing period
6. Match SMW 42–52 & SMW 1–8 to each potato crop year
7. Create the crop-year weather & yield dataset
8. Detrend potato yield using a linear trend
9. Calculate the yield anomaly & create the lag-1 yield feature
10. Prepare candidate features for modelling
11. Perform feature selection
12. Define the ML models
13. Tune the models using 5-fold cross-validation
14. Evaluate the models using leave-one-out cross-validation (LOOCV)
15. Calculate R², RMSE, MAE
16. Compare all models & feature selection methods
17. Identify the best-performing model
18. Extract the selected features of the best model
19. Refit the best model using the selected features
20. Predict detrended potato yield
21. Add the yield trend back to obtain final predicted yield
22. Compare observed & predicted potato yield
23. Prepare the final results table
24. Save the final results & predictions

**Weather parameters used (5 only):** `MAXT`, `MINT`, `MEAN RH`, `WS` (wind speed), `BSS` (bright sunshine hours).
**Rainfall / precipitation is excluded**, as requested. With 19 season weeks this gives **5 × 19 = 95** weekly weather features, plus **1 lag-1 yield-anomaly feature** = **96 candidate features** in total.

**Feature-selection methods** (Forward Selection, Backward Elimination, RFE, SelectKBest F-test, Mutual Information) each return between **5 and 10 features** (`K_MIN = 5`, `K_MAX = 10`); the best size is chosen by LOOCV.

**Models:** Stepwise Linear Regression, Random Forest, XGBoost, SVR, KNN, ANN (MLP) — each is first **tuned with 5-fold cross-validation**, then evaluated with **LOOCV**.

---

### Data-leakage fix (read this first)

A review of this notebook found that **feature selection and hyperparameter tuning were being
performed once on the entire dataset** (all crop years, including the year later "held out" by
LOOCV) before the leave-one-out evaluation loop. That lets the model selection step see each
held-out year's own yield, which optimistically biases the reported R²/RMSE/MAE.

This version fixes that, **without changing the modelling workflow, the models, the feature
pool, or the feature-selection methods**:

- **Feature selection (Step 11)** is now re-run *inside every LOOCV fold*, using only that
  fold's training years. The held-out year never influences which weather weeks are chosen.
- **Hyperparameter tuning (Step 13)** is likewise re-run *inside every LOOCV fold*, on only
  that fold's training years, using a nested inner K-fold CV.
- The **reported R²/RMSE/MAE (Steps 14-17)** therefore come entirely from genuinely
  out-of-sample predictions.
- A **separate, full-data feature selection + hyperparameter tuning** is still done once, but
  it is used *only* to build the final production model in Steps 18-21 (the "refit on all
  available data" step) — never to compute the performance metrics. This mirrors normal
  practice: evaluate honestly with nested CV, then refit the chosen recipe on all the data you
  have for the model you'll actually use going forward.
- The **linear yield-detrending step (Step 8)** is intentionally left as a single fit on the
  whole time series, matching the workflow diagram (detrending happens once, before the
  train/test split is ever introduced) and standard practice in agro-meteorological yield
  studies. This is a much smaller effect than the feature-selection leakage and changing it
  would alter the workflow itself, which was not requested.
- Comparing 5 feature-selection methods x 6 values of *k* x 6 models and keeping the best
  combination is part of the intended workflow (Steps 16-17), not a bug — it's now just being
  scored honestly rather than on leaked data.

Everything else (the models, the 5 feature-selection methods, K_MIN/K_MAX, the final results
table, and the saved output files) is unchanged.


## 1. Install required python libraries

In [ ]:
# Install the packages this notebook needs (Colab already has most of these,
# but we pin/ensure xgboost, statsmodels and openpyxl are present).
!pip install -q xgboost statsmodels openpyxl
print("Libraries installed.")

## 2. Upload the weather & potato yield data files

In [ ]:
import glob, os

try:
    from google.colab import files
    print("Please upload BOTH files: the productivity (yield) xlsx and the weather xlsx")
    uploaded = files.upload()
    names = list(uploaded.keys())
except ImportError:                       # running locally / Jupyter instead of Colab
    names = glob.glob("*.xlsx")

YIELD_PATH   = [f for f in names if "Productivity" in f or "productivity" in f.lower()][0]
WEATHER_PATH = [f for f in names if "Weather" in f or "weather" in f.lower()][0]
print("Yield file   :", YIELD_PATH)
print("Weather file :", WEATHER_PATH)

## 3. Import libraries & set up analysis

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.feature_selection import RFE, f_regression, mutual_info_regression, SelectKBest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneOut, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

K_MIN, K_MAX = 5, 10          # number of selected features must lie in this range
print("Setup complete. K_MIN =", K_MIN, " K_MAX =", K_MAX)

## 4. Load daily weather data & convert it to SMW
SMW 1 = Jan 1–7, …, SMW 52 = last week (absorbs the extra day(s) of the year).

**Only 5 weather parameters are used** — `BSS`, `WS` (wind speed), `MAXT`, `MINT`, `MEAN RH` — **rainfall/precipitation is excluded**.

In [ ]:
wx = pd.read_excel(WEATHER_PATH, sheet_name="Daily_Weather")
wx["DATE"] = pd.to_datetime(wx["DATE"])
wx["year"] = wx["DATE"].dt.year
wx["doy"]  = wx["DATE"].dt.dayofyear
wx["smw"]  = np.minimum(((wx["doy"] - 1) // 7) + 1, 52)

# parameter name used in the reference table -> (column in file, weekly aggregation)
# NOTE: Precipitation / rainfall is intentionally EXCLUDED -> only 5 parameters
PARAMS = {
    "BSS":     ("BSS_Sunshine_Hours_per_day", "mean"),
    "WS":      ("Wind_Speed_10m_m_s",        "mean"),
    "MAXT":    ("Max_Temperature_C",         "mean"),
    "MINT":    ("Min_Temperature_C",         "mean"),
    "MEAN RH": ("Relative_Humidity_percent", "mean"),
}
print(f"Weather data spans {wx['DATE'].min().date()} to {wx['DATE'].max().date()}  ({len(wx)} daily records)")
print(f"Weather parameters used ({len(PARAMS)}): {list(PARAMS.keys())}")
wx.head()

## 5. Load potato yield data & define the potato growing period
The potato growing (Rabi) season is defined as **SMW 42–52** (mid-Oct to end-Dec, in the crop year's start calendar year) plus **SMW 1–8** (Jan–Feb, in the following calendar year) → **19 season weeks** per crop year.

In [ ]:
yld_raw = pd.read_excel(YIELD_PATH, skiprows=1)
yld_raw.columns = ["Year", "Productivity"]
yld_raw = yld_raw.dropna(subset=["Year"])
yld_raw = yld_raw[yld_raw["Year"].astype(str).str.match(r"^\d{4}-\d{2}$")].copy()
yld_raw["StartYear"] = yld_raw["Year"].str[:4].astype(int)
yld_raw["Productivity"] = pd.to_numeric(yld_raw["Productivity"], errors="coerce")
yld_raw = yld_raw.reset_index(drop=True)
print(f"Loaded {len(yld_raw)} crop-year yield records ({yld_raw['Year'].iloc[0]} to {yld_raw['Year'].iloc[-1]})")

# Potato growing period definition: SMW 42-52 (year Y) + SMW 1-8 (year Y+1) = 19 weeks
SEASON_WEEKS = list(range(42, 53)) + list(range(1, 9))
print(f"Growing period defined: SMW {SEASON_WEEKS[0]}-{SEASON_WEEKS[10]} and SMW {SEASON_WEEKS[11]}-{SEASON_WEEKS[-1]}  "
      f"-> {len(SEASON_WEEKS)} season weeks")
yld_raw.head()

## 6. Match SMW 42–52 & SMW 1–8 to each potato crop year
For crop year **Y** (e.g. "1996-97"): SMW 42–52 of calendar year **Y** + SMW 1–8 of calendar year **Y+1**.
Each weekly weather value is matched to its crop year, giving `PARAM_wNN` columns, e.g. `BSS_w36`, `MINT_w1` (5 parameters × 19 weeks = **95** weekly weather features).

In [ ]:
def season_weekly_features(start_year, weather_df):
    """Match SMW 42-52 (calendar year = start_year) and SMW 1-8 (calendar year = start_year+1)
    weather to a single potato crop year."""
    out = {}
    for smw in SEASON_WEEKS:
        cal_year = start_year if smw >= 42 else start_year + 1
        wk = weather_df[(weather_df["year"] == cal_year) & (weather_df["smw"] == smw)]
        for pname, (col, agg) in PARAMS.items():
            out[f"{pname}_w{smw}"] = wk[col].sum() if agg == "sum" else wk[col].mean()
    return out

wx_min_year, wx_max_year = wx["year"].min(), wx["year"].max()
usable_years = [sy for sy in yld_raw["StartYear"] if sy >= wx_min_year and (sy + 1) <= wx_max_year]
print(f"Usable crop years: {usable_years[0]}-{str(usable_years[0]+1)[-2:]} to "
      f"{usable_years[-1]}-{str(usable_years[-1]+1)[-2:]}  -> n = {len(usable_years)}")

rows = [{"StartYear": sy, **season_weekly_features(sy, wx)} for sy in usable_years]
feat_df = pd.DataFrame(rows)
print("Matched weekly-weather rows:", feat_df.shape)
feat_df.head()

## 7. Create the crop-year weather & yield dataset

In [ ]:
data = feat_df.merge(yld_raw[["StartYear", "Year", "Productivity"]], on="StartYear") \
              .sort_values("StartYear").reset_index(drop=True)

WEEKLY_FEATURES = [c for c in data.columns if c not in ("StartYear", "Year", "Productivity")]
assert data[WEEKLY_FEATURES].isna().sum().sum() == 0, "NaNs in weekly features - check weather coverage"
assert len(WEEKLY_FEATURES) == len(PARAMS) * len(SEASON_WEEKS)
print(f"Weekly weather features: {len(WEEKLY_FEATURES)}  ({len(PARAMS)} parameters x {len(SEASON_WEEKS)} weeks)")

data.to_csv("season_weekly_features.csv", index=False)
data[["StartYear", "Year", "Productivity"] + WEEKLY_FEATURES[:6]].head()

## 8. Detrend potato yield using a linear trend

In [ ]:
y_full = data["Productivity"].reset_index(drop=True)
t_full = np.arange(len(data)).reshape(-1, 1)

trend_model = LinearRegression().fit(t_full, y_full)
trend_full = trend_model.predict(t_full)
resid_full = y_full - trend_full            # weather-driven residual (yield anomaly)

print(f"Trend line: Yield = {trend_model.coef_[0]:.4f} * t + {trend_model.intercept_:.4f}")
print(f"Trend R2: {trend_model.score(t_full, y_full):.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(data["StartYear"], y_full, "o-", label="Actual yield")
ax.plot(data["StartYear"], trend_full, "--", label="Fitted trend")
ax.set_xlabel("Crop year (start year)"); ax.set_ylabel("Productivity (MT/ha)")
ax.set_title("Potato yield: actual vs. long-term trend")
ax.legend(); plt.tight_layout(); plt.show()

## 9. Calculate the yield anomaly & create the lag-1 yield feature
The **yield anomaly** is the detrended residual from the previous step. We also create a **lag-1 yield anomaly** feature
(the previous crop year's yield anomaly), since last year's yield can carry information about this year's yield.
Because the very first crop year has no prior year available, that row is dropped from the modelling dataset.

In [ ]:
data["YIELD_ANOMALY"] = resid_full.values
data["YIELD_LAG1"] = data["YIELD_ANOMALY"].shift(1)

# Drop the first crop year (no lag-1 value available)
model_data = data.dropna(subset=["YIELD_LAG1"]).reset_index(drop=True)
print(f"Crop years available for modelling: {len(model_data)}  "
      f"({model_data['Year'].iloc[0]} to {model_data['Year'].iloc[-1]})  "
      f"[1 year dropped because it has no lag-1 yield]")

model_data[["StartYear", "Year", "Productivity", "YIELD_ANOMALY", "YIELD_LAG1"]].head()

## 10. Prepare candidate features for modelling
Candidate feature pool = **95 weekly weather features + 1 lag-1 yield-anomaly feature = 96 candidates**.

In [ ]:
FEATURES = WEEKLY_FEATURES + ["YIELD_LAG1"]
assert len(FEATURES) == len(PARAMS) * len(SEASON_WEEKS) + 1

X_all = model_data[FEATURES].reset_index(drop=True)
resid = model_data["YIELD_ANOMALY"].reset_index(drop=True)     # target for the weather models
y      = model_data["Productivity"].reset_index(drop=True)     # actual yield, for these crop years
trend  = (model_data["Productivity"] - model_data["YIELD_ANOMALY"]).values   # trend value for these crop years

assert X_all.isna().sum().sum() == 0, "NaNs in candidate features"
print(f"Candidate features: {len(FEATURES)}  "
      f"({len(PARAMS)} weather parameters x {len(SEASON_WEEKS)} weeks + 1 lag-1 yield feature)")
print("X_all shape:", X_all.shape)

## 11. Perform feature selection
With 96 candidates and a small sample of crop years, each method produces a ranked / nested list and the subset of size
**k ∈ [5, 10]** is chosen later by LOOCV.

| Method | How the 5–10 features are obtained |
|---|---|
| Forward Selection | greedily add the feature that most improves LOOCV R² (linear model) until 10; the first k form the size-k subset |
| Backward Elimination | start from the 20 strongest F-test features, repeatedly drop the feature whose removal gives the best LOOCV R², down to 5 |
| RFE | recursive feature elimination (Ridge estimator) to exactly k features |
| SelectKBest (F-test) | top-k features by F-score |
| Mutual Information | top-k features by mutual information |

**Leakage fix:** these functions are defined here but are **not** run on the full dataset for evaluation
purposes. They are called again, fresh, inside every LOOCV fold in Step 14 (using only that fold's
training years), and once more on the full dataset near Step 18 purely to build the final production
model. The functions were already written to accept any `X`/`target`, so no methodology changed — only
*when* and *on what data* they are called.

In [ ]:
def _press_r2(Xs, target):
    """Exact LOOCV R2 of an OLS model via the hat-matrix (PRESS) shortcut - very fast."""
    A = np.column_stack([np.ones(len(Xs)), Xs])
    H = A @ np.linalg.pinv(A.T @ A) @ A.T
    e = target - H @ target
    loo_res = e / (1 - np.clip(np.diag(H), None, 0.999999))
    return 1 - np.sum(loo_res ** 2) / np.sum((target - target.mean()) ** 2)


def forward_path(Xs, target, cols, kmax=K_MAX):
    sel, remaining = [], list(range(Xs.shape[1]))
    while len(sel) < kmax:
        scores = [(_press_r2(Xs[:, sel + [j]], target), j) for j in remaining]
        best_j = max(scores)[1]
        sel.append(best_j); remaining.remove(best_j)
    return [cols[j] for j in sel]                 # ordered list; prefix of length k = size-k subset


def backward_subsets(Xs, target, cols, kmin=K_MIN, kmax=K_MAX, pool=20):
    F, _ = f_regression(Xs, target)
    cur = list(np.argsort(F)[::-1][:pool])        # pool of strongest candidates
    subsets = {}
    while len(cur) > kmin:
        scores = [(_press_r2(Xs[:, [c for c in cur if c != d]], target), d) for d in cur]
        drop = max(scores)[1]
        cur.remove(drop)
        if len(cur) <= kmax:
            subsets[len(cur)] = [cols[j] for j in cur]
    return subsets


def rfe_subset(Xs, target, cols, k):
    rfe = RFE(Ridge(alpha=1.0), n_features_to_select=k, step=1).fit(Xs, target)
    return [c for c, s in zip(cols, rfe.support_) if s]


def select_subsets(method, X, target):
    """Return {k: [feature names]} for k = K_MIN..K_MAX using the given method.
    Scaling is done here using only the rows passed in, so this function is safe to call
    with just a fold's training rows (see Step 14) as well as with the full dataset
    (used only for the final production refit near Step 18)."""
    cols = list(X.columns)
    Xs = StandardScaler().fit_transform(X.values)
    tv = np.asarray(target, dtype=float)
    ks = range(K_MIN, K_MAX + 1)
    if method == "Forward Selection":
        path = forward_path(Xs, tv, cols)
        return {k: path[:k] for k in ks}
    if method == "Backward Elimination":
        return backward_subsets(Xs, tv, cols)
    if method == "RFE":
        return {k: rfe_subset(Xs, tv, cols, k) for k in ks}
    if method == "SelectKBest (F-test)":
        F, _ = f_regression(Xs, tv)
        order = [cols[j] for j in np.argsort(F)[::-1]]
        return {k: order[:k] for k in ks}
    if method == "Mutual Information":
        mi = mutual_info_regression(Xs, tv, n_neighbors=3, random_state=RANDOM_STATE)
        order = [cols[j] for j in np.argsort(mi)[::-1]]
        return {k: order[:k] for k in ks}
    raise ValueError(method)


FS_METHODS = ["Forward Selection", "Backward Elimination", "RFE", "SelectKBest (F-test)", "Mutual Information"]
print("Feature-selection methods defined:", FS_METHODS)
print("NOTE: not executed on the full dataset here — see Step 14 (nested, per-fold) and Step 18 (final refit).")

In [ ]:
def format_features(feats):
    """Turn ['BSS_w12','BSS_w36','WS_w6', 'YIELD_LAG1', ...] into a readable, grouped string,
    e.g. 'BSS (36, 12)\nWS (6)\nLag-1 Yield Anomaly', with weeks in season order."""
    grouped = {}
    has_lag = False
    for f in feats:
        if f == "YIELD_LAG1":
            has_lag = True
            continue
        p, w = f.rsplit("_w", 1)
        grouped.setdefault(p, []).append(int(w))
    parts = []
    for p in PARAMS:                                   # fixed parameter order
        if p in grouped:
            wks = sorted(grouped[p], key=SEASON_WEEKS.index)   # season order 42..52, 1..8
            parts.append(f"{p} ({', '.join(map(str, wks))})")
    if has_lag:
        parts.append("Lag-1 Yield Anomaly")
    return "\n".join(parts)


# Illustration only: run on a subset that EXCLUDES the most recent crop year, exactly the way a
# single LOOCV training fold would look. This is just to preview the output format and is not
# used anywhere else below (the real, per-fold selections happen fresh inside the nested LOOCV
# loop in Step 14).
_illustration_rows = np.arange(len(X_all) - 1)
_demo_subsets = select_subsets("Forward Selection", X_all.iloc[_illustration_rows], resid.iloc[_illustration_rows])
for _k, _feats in _demo_subsets.items():
    assert K_MIN <= len(_feats) <= K_MAX
print(f"Illustrative subset sizes (Forward Selection, training-only rows): {sorted(_demo_subsets)}")
print()
print(format_features(_demo_subsets[K_MAX]))

## 12. Define the ML models
Six regressors: **Stepwise Linear Regression, Random Forest, XGBoost, SVR, KNN, ANN (MLP)**.

In [ ]:
def make_estimator(name, params=None):
    """Build a fresh estimator (or scaling pipeline). `params` overrides the default
    hyperparameters - used after tuning (Step 13) to plug in the tuned values."""
    params = params or {}
    if name == "Stepwise Linear Regression":
        return lambda: LinearRegression()
    if name == "Random Forest":
        p = {"n_estimators": 150, "max_depth": 4, **params}
        return lambda: RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **p)
    if name == "XGBoost":
        p = {"n_estimators": 150, "max_depth": 3, "learning_rate": 0.05,
             "subsample": 0.8, "colsample_bytree": 0.8, **params}
        return lambda: XGBRegressor(random_state=RANDOM_STATE, verbosity=0, n_jobs=-1, **p)
    if name == "SVR":
        p = {"C": 10, "epsilon": 0.3, **params}
        return lambda: Pipeline([("sc", StandardScaler()), ("svr", SVR(kernel="rbf", **p))])
    if name == "KNN":
        p = {"n_neighbors": 5, **params}
        return lambda: Pipeline([("sc", StandardScaler()), ("knn", KNeighborsRegressor(**p))])
    if name == "ANN (MLP)":
        p = {"hidden_layer_sizes": (8,), "alpha": 0.0001, **params}
        return lambda: Pipeline([("sc", StandardScaler()),
                                 ("mlp", MLPRegressor(activation="relu", solver="lbfgs", max_iter=3000,
                                                      random_state=RANDOM_STATE, **p))])
    raise ValueError(name)


def r2_rmse(actual, pred):
    actual = np.asarray(actual, dtype=float)
    pred = np.asarray(pred, dtype=float)
    ss_res = np.sum((actual - pred) ** 2)
    r2 = 1 - ss_res / np.sum((actual - actual.mean()) ** 2)
    rmse = np.sqrt(ss_res / len(actual))
    mae = np.mean(np.abs(actual - pred))
    return r2, rmse, mae


model_names = ["Stepwise Linear Regression", "Random Forest", "XGBoost", "SVR", "KNN", "ANN (MLP)"]
print("Models defined:", model_names)

## 13. Tune the models using 5-fold cross-validation
Each model's hyperparameters are tuned with `GridSearchCV` (5-fold CV, or fewer folds if a training
set is small) on a representative feature set (the top-10 candidate features by F-test score).
Stepwise Linear Regression has no hyperparameters to tune, so it is left as-is.

**Leakage fix:** tuning is now performed *twice*, separately:
1. **Inside every LOOCV fold** (Step 14), using only that fold's training years — this is what the
   honest R²/RMSE/MAE are based on.
2. **Once here, on the full dataset** — used *only* to build the final production model in Steps
   18-21. These full-data hyperparameters never feed into the performance metrics.

In [ ]:
PARAM_GRIDS = {
    "Random Forest": {"n_estimators": [100, 150, 200], "max_depth": [3, 4, 5]},
    "XGBoost":        {"n_estimators": [100, 150], "max_depth": [2, 3, 4], "learning_rate": [0.03, 0.05, 0.1]},
    "SVR":            {"svr__C": [1, 10, 50], "svr__epsilon": [0.1, 0.3, 0.5]},
    "KNN":            {"knn__n_neighbors": [3, 5, 7]},
    "ANN (MLP)":      {"mlp__hidden_layer_sizes": [(8,), (16,), (8, 4)], "mlp__alpha": [0.0001, 0.001, 0.01]},
}


def tune_hyperparams_on(name, X_tr, resid_tr, random_state=RANDOM_STATE, rep_k=10, n_inner=5):
    """Tune one model's hyperparameters using ONLY the rows passed in (X_tr / resid_tr).
    Reused both for the nested, per-fold tuning in Step 14 (leakage-free evaluation) and for
    the one-off full-data tuning below (used only for the final production refit)."""
    if name not in PARAM_GRIDS:
        return {}
    kk = min(rep_k, X_tr.shape[1])
    skb = SelectKBest(f_regression, k=kk).fit(X_tr.values, resid_tr.values)
    tune_cols = [c for c, s in zip(X_tr.columns, skb.get_support()) if s]
    X_tune = StandardScaler().fit_transform(X_tr[tune_cols].values)
    n_splits = max(2, min(n_inner, len(X_tr)))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    base_fn = make_estimator(name)
    gs = GridSearchCV(base_fn(), PARAM_GRIDS[name], cv=kf, scoring="r2", n_jobs=-1)
    gs.fit(X_tune, resid_tr.values)
    return {k.split("__")[-1]: v for k, v in gs.best_params_.items()}


# Full-data tuning: used ONLY for the final production refit in Steps 18-21, never for scoring.
TUNED_PARAMS_FULL = {}
for name in model_names:
    TUNED_PARAMS_FULL[name] = tune_hyperparams_on(name, X_all, resid)
    print(f"{name:28s} full-data tuned params (final refit only): {TUNED_PARAMS_FULL[name]}")


def make_estimator_tuned_full(name):
    return make_estimator(name, TUNED_PARAMS_FULL.get(name, {}))

## 14. Evaluate the models using leave-one-out cross-validation (LOOCV)
## 15. Calculate R², RMSE, MAE
Every model is evaluated by **nested** LOOCV across all 5 feature-selection methods and all
k = 5…10 subsets (on the weather-driven yield residual). "Nested" means that for every held-out
crop year, **feature selection and hyperparameter tuning are both re-fit using only the
remaining (training) years** — the held-out year is never used to choose features or tune
hyperparameters, only to test the already-frozen model. The (method, k) combination with the
best nested-LOOCV R² is kept for each model. This is slower than fitting feature selection once
on the full dataset, but it removes the data-leakage flagged in the review.

In [ ]:
def loocv_nested_predict_all():
    """Nested LOOCV: for every held-out crop year, feature selection AND hyperparameter tuning
    are refit using ONLY the remaining (training) years, so the held-out year never influences
    which features or hyperparameters are used to predict it. Same models, same 5
    feature-selection methods, same k = 5..10 search as before - just leakage-free."""
    n = len(X_all)
    preds = {name: {m: {k: np.full(n, np.nan) for k in range(K_MIN, K_MAX + 1)} for m in FS_METHODS}
             for name in model_names}

    loo = LeaveOneOut()
    for fold_num, (tr_idx, te_idx) in enumerate(loo.split(X_all), start=1):
        X_tr, X_te = X_all.iloc[tr_idx], X_all.iloc[te_idx]
        resid_tr = resid.iloc[tr_idx]

        # --- feature selection using ONLY this fold's training years ---
        fold_candidate_sets = {m: select_subsets(m, X_tr, resid_tr) for m in FS_METHODS}

        # --- hyperparameter tuning using ONLY this fold's training years ---
        fold_tuned = {name: tune_hyperparams_on(name, X_tr, resid_tr) for name in model_names}

        # --- fit every (model, fs_method, k) combo on the training years, predict the held-out year ---
        for name in model_names:
            params = fold_tuned[name]
            for fs_name, by_k in fold_candidate_sets.items():
                for k, feats in by_k.items():
                    est = make_estimator(name, params)()
                    est.fit(X_tr[feats], resid_tr)
                    preds[name][fs_name][k][te_idx] = est.predict(X_te[feats])

        if fold_num % 5 == 0 or fold_num == n:
            print(f"  nested LOOCV fold {fold_num}/{n} complete")

    return preds


print(f"Running nested LOOCV over {len(X_all)} crop years x {len(FS_METHODS)} feature-selection "
      f"methods x {K_MAX - K_MIN + 1} values of k x {len(model_names)} models.")
print("Feature selection and hyperparameter tuning are refit inside every fold -> slower, but leakage-free.")
nested_preds = loocv_nested_predict_all()
print("Nested LOOCV complete.\n")

results = []
for name in model_names:
    best = {"r2": -np.inf}
    for fs_name in FS_METHODS:
        for k in range(K_MIN, K_MAX + 1):
            preds = nested_preds[name][fs_name][k]
            r2, rmse, mae = r2_rmse(resid.values, preds)
            if r2 > best["r2"]:
                best = {"model": name, "fs_method": fs_name, "k": k,
                        "r2": r2, "rmse": rmse, "mae": mae, "loocv_pred_resid": preds}

    # Full-data feature set for the winning (fs_method, k) -- used ONLY for display and for the
    # final production refit in Steps 18-21 (never for the R2/RMSE/MAE above).
    full_sets_for_method = select_subsets(best["fs_method"], X_all, resid)
    best["features"] = full_sets_for_method[best["k"]]

    # Full-data refit (full-data-tuned hyperparameters + full-data-selected features) -> used for
    # the later refit step, same role as in the original notebook.
    full_est_fn = make_estimator_tuned_full(best["model"])
    full_fit = full_est_fn().fit(X_all[best["features"]], resid)
    best["full_pred_resid"] = full_fit.predict(X_all[best["features"]])

    # significance of the (honest, nested) LOOCV predictions
    _, p_val = stats.pearsonr(best["loocv_pred_resid"], resid.values)
    best["p_value"] = p_val
    best["stars"] = "**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns")

    results.append(best)
    print(f"{name:28s} best: {best['fs_method']:22s} k={best['k']:2d}  "
          f"nested-LOOCV R2(resid)={best['r2']:.3f}{best['stars']}  "
          f"RMSE(resid)={best['rmse']:.3f}  MAE(resid)={best['mae']:.3f}")

## 16. Compare all models & feature selection methods

In [ ]:
comparison_rows = []
for r in results:
    comparison_rows.append({
        "Model": r["model"],
        "R2": f"{r['r2']:.2f}{r['stars']}",          # e.g. "0.89**"  (** p<0.01, * p<0.05, ns = not significant)
        "RMSE": round(r["rmse"], 3),
        "MAE": round(r["mae"], 3),
        "Feature Selection": r["fs_method"],
        "No. of features": r["k"],
        "Selected Features (Weather parameters with meteorological week)": format_features(r["features"]),
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df["_r2_sort"] = [r["r2"] for r in results]
comparison_df = comparison_df.sort_values("_r2_sort", ascending=False).drop(columns="_r2_sort").reset_index(drop=True)

pd.set_option("display.max_colwidth", None)
comparison_df

## 17. Identify the best-performing model

In [ ]:
best_idx = int(np.argmax([r["r2"] for r in results]))
best = results[best_idx]

print(f"Best-performing model: {best['model']}")
print(f"  Feature selection method : {best['fs_method']}")
print(f"  Number of features (k)   : {best['k']}")
print(f"  LOOCV R2 (residual)      : {best['r2']:.3f}")
print(f"  RMSE (residual)          : {best['rmse']:.3f}")
print(f"  MAE (residual)           : {best['mae']:.3f}")

## 18. Extract the selected features of the best model

In [ ]:
best_features = best["features"]
print(f"Selected features ({len(best_features)}):\n")
print(format_features(best_features))

## 19. Refit the best model using the selected features

In [ ]:
best_estimator_fn = make_estimator_tuned_full(best["model"])
best_model_refit = best_estimator_fn().fit(X_all[best_features], resid)
print(f"{best['model']} refit on the full dataset using its {len(best_features)} selected features "
      f"and its full-data-tuned hyperparameters (production model; the R2/RMSE/MAE reported above "
      f"came from the leakage-free nested LOOCV, not from this refit).")

## 20. Predict detrended potato yield

In [ ]:
# Full-data (refit) prediction of the detrended yield (weather-driven residual)
pred_detrended_refit = best_model_refit.predict(X_all[best_features])

# LOOCV prediction of the detrended yield (out-of-sample, the honest / realistic estimate)
pred_detrended_loocv = best["loocv_pred_resid"]

print("Predicted detrended yield (residual) computed for both the full refit and LOOCV.")

## 21. Add the yield trend back to obtain final predicted yield

In [ ]:
final_pred_refit = pred_detrended_refit + trend
final_pred_loocv = pred_detrended_loocv + trend

r2_refit, rmse_refit, mae_refit = r2_rmse(y.values, final_pred_refit)
r2_loocv, rmse_loocv, mae_loocv = r2_rmse(y.values, final_pred_loocv)

print(f"Final predicted yield (full refit) : R2 = {r2_refit:.3f}   RMSE = {rmse_refit:.3f} MT/ha   MAE = {mae_refit:.3f} MT/ha")
print(f"Final predicted yield (LOOCV)       : R2 = {r2_loocv:.3f}   RMSE = {rmse_loocv:.3f} MT/ha   MAE = {mae_loocv:.3f} MT/ha")

## 22. Compare observed & predicted potato yield

In [ ]:
_, p_val = stats.pearsonr(pred_detrended_loocv, resid.values)
stars = "**" if p_val < 0.01 else ("*" if p_val < 0.05 else "ns")

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(model_data["StartYear"], y, "o-", label="Observed yield")
ax.plot(model_data["StartYear"], final_pred_refit, "s--", label=f"Predicted ({best['model']}, refit)")
ax.plot(model_data["StartYear"], final_pred_loocv, "^:", label=f"Predicted ({best['model']}, LOOCV)")
ax.set_xlabel("Crop year (start year)"); ax.set_ylabel("Productivity (MT/ha)")
ax.set_title(f"Best model: {best['model']} | {best['k']} features | "
             f"R2(LOOCV)={r2_loocv:.2f}{stars}")
ax.legend(); plt.tight_layout(); plt.show()

print(f"Correlation (LOOCV predicted vs actual residual): p = {p_val:.4f} ({stars})")

## 23. Prepare the final results table

In [ ]:
final_table = pd.DataFrame({
    "StartYear": model_data["StartYear"],
    "Year": model_data["Year"],
    "Observed Yield (MT/ha)": y.values,
    "Predicted Yield - Refit (MT/ha)": np.round(final_pred_refit, 2),
    "Predicted Yield - LOOCV (MT/ha)": np.round(final_pred_loocv, 2),
})

summary_row = {
    "Model": best["model"],
    "Feature Selection": best["fs_method"],
    "No. of features": best["k"],
    "R2 (final yield, refit)": round(r2_refit, 3),
    "R2 (final yield, LOOCV)": round(r2_loocv, 3),
    "RMSE (final yield, refit)": round(rmse_refit, 3),
    "RMSE (final yield, LOOCV)": round(rmse_loocv, 3),
    "MAE (final yield, refit)": round(mae_refit, 3),
    "MAE (final yield, LOOCV)": round(mae_loocv, 3),
    "Selected Features": format_features(best_features),
}
summary_df = pd.DataFrame([summary_row])

pd.set_option("display.max_colwidth", None)
print("Best-model summary:")
display(summary_df)
print("\nModel comparison (all 6 models):")
display(comparison_df)
print("\nPer-year predictions (best model):")
display(final_table)

## 24. Save the final results & predictions

In [ ]:
out_xlsx = "potato_yield_model_results.xlsx"
with pd.ExcelWriter(out_xlsx, engine="openpyxl") as xw:
    summary_df.to_excel(xw, index=False, sheet_name="Best_Model_Summary")
    comparison_df.to_excel(xw, index=False, sheet_name="Model_Comparison")
    final_table.to_excel(xw, index=False, sheet_name="Predictions")

    from openpyxl.styles import Alignment, Font
    for sheet in xw.sheets.values():
        for cell in sheet[1]:
            cell.font = Font(bold=True); cell.alignment = Alignment(wrap_text=True, vertical="top")
        for row in sheet.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(wrap_text=True, vertical="top")

comparison_df.to_csv("model_comparison_results.csv", index=False)
final_table.to_csv("final_predictions.csv", index=False)

print(f"Saved: {out_xlsx}")
print("Saved: model_comparison_results.csv")
print("Saved: final_predictions.csv")

try:
    from google.colab import files as _f
    _f.download(out_xlsx)
except Exception as e:
    print("Auto-download skipped (not running in Colab):", e)